In [1]:
import numpy as np
import sklearn

from utils import load_preprocessed_ds

In [2]:
X_tr, y_tr, X_te, y_te, X_tr_pd = load_preprocessed_ds()

In [3]:
rng = np.random.default_rng(44)

In [4]:
# Boostrap sampling function
def bootstrap_sample(X, y, rng):
    n_samples = X.shape[0]
    sampled_indices = rng.choice(n_samples, size=n_samples, replace=True)

    X_boot = X[sampled_indices]
    y_boot = y[sampled_indices]

    bootstrap_indices = np.unique(sampled_indices)
    all_indices = np.arange(n_samples)
    sample_mask = np.isin(all_indices, bootstrap_indices)
    oob_indices = all_indices[~sample_mask]

    # print(len(oob_indices)/n_samples) # Should be ~ 0.37
    # print(np.intersect1d(bootstrap_indices, oob_indices)) # Should be empty
    return X_boot, y_boot, oob_indices

bootstrap_sample(X_tr, y_tr, rng)

(array([[ 1.69149923,  1.        ,  1.76494012, ...,  1.        ,
          0.        ,  0.        ],
        [ 2.34470585,  0.        ,  0.57404228, ...,  0.        ,
          0.        ,  0.        ],
        [-1.46566609,  1.        , -1.09321469, ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 1.03829261,  1.        , -0.61685555, ...,  1.        ,
          0.        ,  0.        ],
        [-0.26812062,  1.        , -0.14049642, ...,  1.        ,
          0.        ,  0.        ],
        [ 0.7116893 ,  0.        ,  0.87176674, ...,  0.        ,
          0.        ,  0.        ]], shape=(242, 20)),
 array([0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 1., 0., 0., 1.,
        1., 0., 0., 1., 1., 1., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
        1., 0., 1., 0., 1., 1., 1., 0., 1., 0., 0., 1., 1., 1., 1., 1., 0.,
        0., 1., 1., 1., 0., 0., 0., 1., 0., 0., 1., 1., 1., 0., 1., 1., 0.,
        1., 1., 1., 0., 1., 1., 1., 1., 0., 0., 1., 

In [5]:
MIN_SAMPLES_SPLIT = 10
MAX_DEPTH = 3
MAX_FEATURES = 3
N_TREES = 100
SEED = 42

In [6]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.is_leaf = self.value is not None

    def predict_one(self, x):
        if self.is_leaf:
            return self.value
        else:
            if x[self.feature] <= self.threshold:
                return self.left.predict_one(x)
            else:
                return self.right.predict_one(x)

    def predict_batch(self, X):
        return np.array([self.predict_one(X[i,:]) for i in range(X.shape[0])])

def most_common_label(labels):
    unique_vals, counts = np.unique(labels, return_counts=True)
    return unique_vals[np.argmax(counts)]

def split_labels(y, feature, threshold):
    y_left = y[feature <= threshold]
    y_right = y[feature > threshold]
    return y_left, y_right

def gini(y): # y is labels
    labels_count = len(y)
    if labels_count == 0:
        return 0
    _, frequencies = np.unique(y, return_counts=True)
    probabilities = np.sum((frequencies / labels_count) ** 2)
    return 1 - probabilities

def weighted_gini(y_left, y_right):
    len_left = len(y_left)
    len_right = len(y_right)
    len_total = len_left + len_right
    return (len_left * gini(y_left) + len_right * gini(y_right)) / len_total

def find_best_split(X, y, rng, max_features=MAX_FEATURES):
    lowest_gini = np.inf
    feature_at_lowest_gini = None
    thresh_at_lowest_gini = None

    n_features = X.shape[1]
    if max_features > n_features:
        raise Exception("max_features cannot be bigger than n_features!")

    random_features = rng.choice(n_features, size=max_features, replace=False)
    for feature_idx in random_features:

        unique_sorted_vals = np.unique(X[:, feature_idx])
        thresholds = (unique_sorted_vals[:-1] + unique_sorted_vals[1:]) / 2

        for threshold in thresholds:
            y_left, y_right = split_labels(y, X[:,feature_idx], threshold)
            current_gini = weighted_gini(y_left, y_right)

            if current_gini < lowest_gini:
                lowest_gini = current_gini
                feature_at_lowest_gini = feature_idx
                thresh_at_lowest_gini = threshold
    return lowest_gini, feature_at_lowest_gini, thresh_at_lowest_gini

find_best_split(X_tr, y_tr, rng)

def build_tree(X, y, rng, max_features=MAX_FEATURES, depth=0, max_depth=None, min_samples_split=MIN_SAMPLES_SPLIT):
    if min_samples_split < 2:
        raise Exception("min_samples_split cannot be lower than 2!")

    # stopping conditions
    reached_max_depth = (depth == max_depth) if max_depth else False

    too_little_samples_left = len(y) < min_samples_split
    all_samples_same_class = len(np.unique(y)) == 1
    if reached_max_depth or too_little_samples_left or all_samples_same_class:
        return Node(value=most_common_label(y)) # leaf

    lowest_weighted_gini, feature, thresh = find_best_split(X, y, rng, max_features)

    # another stopping condition
    no_improvement_in_splitting = lowest_weighted_gini >= gini(y)
    if no_improvement_in_splitting:
        return Node(value=most_common_label(y)) # leaf

    # split X and y
    mask = X[:, feature] <= thresh
    X_left = X[mask, :]
    y_left = y[mask]
    X_right = X[~mask, :]
    y_right = y[~mask]

    # recurse
    left_node = build_tree(X_left, y_left, rng, max_features, depth + 1, max_depth, min_samples_split)
    right_node = build_tree(X_right, y_right, rng, max_features, depth + 1, max_depth, min_samples_split)

    return Node(feature=feature, threshold=thresh, left=left_node, right=right_node)

In [12]:
# Confirm overfitting when max_features = total features, and max_depth is unlimited
root = build_tree(X_tr, y_tr, rng, max_features=20, depth=0, max_depth=None, min_samples_split=2)

train_preds = root.predict_batch(X_tr)
print(f"Train accuracy: {np.mean(train_preds == y_tr)}")

test_preds = root.predict_batch(X_te)
print(f"Test accuracy: {np.mean(test_preds == y_te)}")

Train accuracy: 1.0
Test accuracy: 0.7540983606557377


In [8]:
def print_tree_sklearn_style(node, feature_names, depth=0):
    indent = "|   " * depth
    if node.is_leaf:
        print(f"{indent}|--- class: {node.value}")
    else:
        fname = feature_names[node.feature]
        print(f"{indent}|--- {fname} <= {node.threshold:.2f}")
        print_tree_sklearn_style(node.left, feature_names, depth + 1)
        print(f"{indent}|--- {fname} >  {node.threshold:.2f}")
        print_tree_sklearn_style(node.right, feature_names, depth + 1)

print_tree_sklearn_style(root, list(X_tr_pd.columns))

|--- thal_2.0 <= 0.50
|   |--- oldpeak <= -0.42
|   |   |--- ca_1.0 <= 0.50
|   |   |   |--- chol <= -0.18
|   |   |   |   |--- chol <= -1.37
|   |   |   |   |   |--- class: 0.0
|   |   |   |   |--- chol >  -1.37
|   |   |   |   |   |--- class: 1.0
|   |   |   |--- chol >  -0.18
|   |   |   |   |--- cp_2.0 <= 0.50
|   |   |   |   |   |--- chol <= 0.92
|   |   |   |   |   |   |--- class: 0.0
|   |   |   |   |   |--- chol >  0.92
|   |   |   |   |   |   |--- class: 1.0
|   |   |   |   |--- cp_2.0 >  0.50
|   |   |   |   |   |--- class: 1.0
|   |   |--- ca_1.0 >  0.50
|   |   |   |--- age <= -0.32
|   |   |   |   |--- class: 1.0
|   |   |   |--- age >  -0.32
|   |   |   |   |--- class: 0.0
|   |--- oldpeak >  -0.42
|   |   |--- thalach <= -0.23
|   |   |   |--- thal_3.0 <= 0.50
|   |   |   |   |--- slope_2.0 <= 0.50
|   |   |   |   |   |--- trestbps <= -1.09
|   |   |   |   |   |   |--- class: 1.0
|   |   |   |   |   |--- trestbps >  -1.09
|   |   |   |   |   |   |--- class: 0.0
|   |   |

In [9]:
class RandomForest:
    def __init__(self, n_trees, max_features, max_depth=None, min_samples_split=MIN_SAMPLES_SPLIT, random_state=None):
        self.n_trees = n_trees
        self.max_features = max_features
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.rng = np.random.default_rng(random_state)
        self.trees = []
        self.oob_indices_per_tree = []

    def fit(self, X, y):
        for _ in range(self.n_trees):
            X_boot, y_boot, oob_idx = bootstrap_sample(X, y, self.rng)
            tree_root = build_tree(X_boot, y_boot, rng=self.rng, max_features=self.max_features, max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            self.trees.append(tree_root)
            self.oob_indices_per_tree.append(oob_idx)

    def predict(self, X):
        all_preds = np.array([tree.predict_batch(X) for tree in self.trees])
        majority_vote = np.apply_along_axis(func1d=most_common_label, axis=0, arr=all_preds) # TODO: vectorize
        return majority_vote

In [19]:
def train_forest_get_acc(mode, n_trees, max_features, random_state):
    if mode == "custom":
        forest = RandomForest(n_trees=n_trees, max_features=max_features, random_state=random_state)
    elif mode == "sklearn":
        forest = sklearn.ensemble.RandomForestClassifier(n_estimators=n_trees, max_features=max_features,random_state=random_state)
    else:
        raise Exception("mode can be either 'custom' or 'sklearn'")

    forest.fit(X_tr, y_tr)
    preds = forest.predict(X_te)
    accuracy = np.mean(preds == y_te)
    return accuracy

In [22]:
# Compare with sklearn
custom_accs = list()
sklearn_accs = list()
for i in range(20):
    custom_accs.append(train_forest_get_acc("custom", n_trees=N_TREES, max_features=MAX_FEATURES, random_state=None))
    sklearn_accs.append(train_forest_get_acc("sklearn", n_trees=N_TREES, max_features=MAX_FEATURES, random_state=None))

custom_accs = np.array(custom_accs)
sklearn_accs = np.array(sklearn_accs)
print(f"custom: {custom_accs.mean():.4f} ± {custom_accs.std():.4f}")
print(f"sklearn: {sklearn_accs.mean():.4f} ± {sklearn_accs.std():.4f}")

custom:  0.8262 ± 0.0190
sklearn: 0.8369 ± 0.0190


In [11]:
# Confirm that 2 forests have the same trees if they use the same seed
forest1 = RandomForest(N_TREES, MAX_FEATURES, random_state=SEED)
forest1.fit(X_tr, y_tr)

forest2 = RandomForest(N_TREES, MAX_FEATURES, random_state=SEED)
forest2.fit(X_tr, y_tr)

assert forest1.trees[0].feature == forest2.trees[0].feature